imports all the tools needed to build a local RAG system.

In [18]:
%pip install langchain langchain-community langchain-core
%pip install chromadb sentence-transformers
%pip install wikipedia-api
%pip install ollama       # For local LLM (Ollama)
%pip install langchain-ollama   


Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


imports libraries needed to build a Retrieval-Augmented Generation (RAG) system.
It handles loading text data, splitting it into chunks, cleaning it, creating embeddings, storing them in ChromaDB, and using Ollama (like Llama 3.2) as the local LLM for answering questions.

In [19]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.llms import Ollama
from langchain_community.document_loaders import TextLoader
import os
import wikipediaapi
import re
from langchain_ollama import OllamaLLM



 fetches Wikipedia pages on quantum computing topics using the Wikipedia API.
Each topic is saved as a cleaned .txt file in a quantum_wiki folder for later use in the RAG system.

In [20]:

wiki = wikipediaapi.Wikipedia(language='en', user_agent='MyRAGApp/1.0')

topics = [
    "Quantum computing", "Qubit", "Superposition", "Quantum entanglement",
    "Quantum measurement", "Quantum decoherence", "Quantum logic gate",
    "Shor's algorithm", "Grover's algorithm", "Quantum Fourier transform",
    "Simon's algorithm", "Quantum error correction", "Surface code",
    "Superconducting qubit", "Trapped ion quantum computer", "Spin qubit",
    "Topological qubit", "Quantum annealing", "Quantum key distribution",
    "Post quantum cryptography", "Quantum teleportation", "Quantum network",
    "Hilbert space", "Pauli matrices", "Dirac notation", "Tensor product",
    "Density matrix", "Unitary operator", "Quantum mechanics",
    "Quantum field theory", "Quantum tunneling", "Quantum thermodynamics",
    "Quantum computing hardware", "Quantum supremacy", "Majorana fermion",
    "Topological quantum computer", "Entanglement entropy"
]

if not os.path.exists("quantum_wiki"):
    os.makedirs("quantum_wiki")

for topic in topics:
    page = wiki.page(topic)
    if page.exists():
        filename = topic.replace(" ", "_") + ".txt"
        with open(f"quantum_wiki/{filename}", "w", encoding="utf-8") as f:
            f.write(page.text)
        print("Saved", filename)
    else:
        print("Page not found:", topic)


Saved Quantum_computing.txt
Saved Qubit.txt
Saved Superposition.txt
Saved Quantum_entanglement.txt
Saved Quantum_measurement.txt
Saved Quantum_decoherence.txt
Saved Quantum_logic_gate.txt
Saved Shor's_algorithm.txt
Saved Grover's_algorithm.txt
Saved Quantum_Fourier_transform.txt
Saved Simon's_algorithm.txt
Saved Quantum_error_correction.txt
Saved Surface_code.txt
Saved Superconducting_qubit.txt
Saved Trapped_ion_quantum_computer.txt
Saved Spin_qubit.txt
Saved Topological_qubit.txt
Saved Quantum_annealing.txt
Saved Quantum_key_distribution.txt
Saved Post_quantum_cryptography.txt
Saved Quantum_teleportation.txt
Saved Quantum_network.txt
Saved Hilbert_space.txt
Saved Pauli_matrices.txt
Saved Dirac_notation.txt
Saved Tensor_product.txt
Saved Density_matrix.txt
Saved Unitary_operator.txt
Saved Quantum_mechanics.txt
Saved Quantum_field_theory.txt
Saved Quantum_tunneling.txt
Saved Quantum_thermodynamics.txt
Page not found: Quantum computing hardware
Saved Quantum_supremacy.txt
Saved Majorana_

 cleans all downloaded Wikipedia .txt files by removing citations like [1], HTML tags, extra spaces, and blank lines.
It overwrites each file with the cleaned version to prepare the dataset for RAG.

In [21]:

folder = "quantum_wiki"

def clean_text(text):
    text = re.sub(r'\[\d+\]', '', text)  # Remove [1], [23], etc
    text = re.sub(r'<.*?>', '', text)  # Remove any HTML-like tags
    text = re.sub(r'\n+', '\n', text)  # Remove blank lines
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    return text.strip()

for file in os.listdir(folder):
    if file.endswith(".txt"):
        path = os.path.join(folder, file)
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        cleaned = clean_text(content)
        with open(path, "w", encoding="utf-8") as f:
            f.write(cleaned)

print("Dataset cleaned successfully.")


Dataset cleaned successfully.


In [22]:

folder = "quantum_wiki"

for file in os.listdir(folder):
    if file.endswith(".txt"):
        path = os.path.join(folder, file)
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()

        # Check for common issues
        has_html = bool(re.search(r'<.*?>', content))
        has_citations = bool(re.search(r'\[\d+\]', content))

        

        if has_html or has_citations:
            print(f"⚠ Still dirty: {file}")

        
def clean_text(text):
    # Remove citation numbers like [1], [23]
    text = re.sub(r'\[\d+\]', '', text)

    # Remove HTML-like tags
    text = re.sub(r'<.*?>', '', text)

    # Remove LaTeX-style expressions { \displaystyle ... }
    text = re.sub(r'\{\\displaystyle.*?\}', '', text)

    # Remove super/subscripts and weird math formatting
    text = re.sub(r'\\[a-zA-Z]+', '', text)  # remove \frac, \sum, etc
    text = re.sub(r'\|.*?\⟩', '', text)  # remove kets like |x⟩

    # Remove unnecessary parentheses blocks
    text = re.sub(r'\{.*?\}', '', text)

    # Replace multiple newlines and spaces with single space
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()



⚠ Still dirty: Hilbert_space.txt


This code loads all cleaned .txt files from the quantum_wiki folder using TextLoader.
It stores their contents in a list called documents, which will be used for chunking and embedding.

In [23]:


folder_path = "quantum_wiki"
documents = []

for file in os.listdir(folder_path):
    if file.endswith(".txt"):
        path = os.path.join(folder_path, file)
        loader = TextLoader(path, encoding="utf-8")
        documents.extend(loader.load())

print("Files loaded:", len(documents))



Files loaded: 36


 splits the loaded documents into smaller chunks using RecursiveCharacterTextSplitter.
Each chunk is up to 1200 characters with a 200-character overlap, ensuring better context retention.
It then prints the total number of chunks created.

In [24]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))


Total chunks created: 1159


This code loads all .txt files from the quantum_wiki folder.
Each file is read using TextLoader, and its content is stored in a list called documents.

In [25]:


folder_path = "quantum_wiki"
documents = []

for file in os.listdir(folder_path):
    if file.endswith(".txt"):
        path = os.path.join(folder_path, file)
        loader = TextLoader(path, encoding="utf-8")
        documents.extend(loader.load())

print("Files loaded:", len(documents))



Files loaded: 36


In [26]:
# Initialize your local model
llm = OllamaLLM(model="llama3.2")


 loads your saved quantum vector database using Chroma and retrieves relevant text chunks using embeddings.
It connects to the Ollama LLM and sends a prompt with the retrieved context to answer user questions.

In [27]:


# Load the database
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = Chroma(
    persist_directory="quantum_vector_db",
    embedding_function=embedding_model
)


# Manual RAG (no RetrievalQA needed)
def rag_query(question):
    # 1 retrieve
    results = db.similarity_search(question, k=5)
    context = "\n".join([doc.page_content for doc in results])
    
    # 2 generate
    prompt = f"Use the context below to answer clearly.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"
    answer = llm.invoke(prompt)
    return answer

# Test
print(rag_query("What is quantum entanglement?"))


Quantum entanglement is the phenomenon where the quantum state of each particle in a group cannot be described independently of the state of the others, even when the particles are separated by a large distance. It is a primary feature of quantum mechanics not present in classical mechanics and can result in seemingly paradoxical effects, such as wave function collapse and changes to the original quantum state upon measurement of a particle's properties.


This function retrieves relevant text from the Chroma vector database and uses it to generate an answer with an LLM.
It formats the context, asks the question, and returns both the answer and the document sources used.

In [28]:

def rag_with_sources(question, k=3):
    retrieved_docs = db.similarity_search(question, k=k)

    used_context = ""
    for i, doc in enumerate(retrieved_docs, start=1):
        used_context += f"\n[Document {i}]\n{doc.page_content}\n"

        prompt = f"""
        You are a RAG system trained on quantum computing and related data from wikipedia.

        Use ONLY the provided context to answer the question.
        - Give a full sentence explanation.
        - Do NOT answer with one word.
        - Cite the document sources used in your answer.
        - Do NOT include unnecessary symbols like |ψ⟩, ⊕, {{}}, or raw LaTeX.
        - If the context is insufficient, say "I don't know."

        Context:
        {used_context}

        Question: {question}
        Answer:
        """


    answer = llm.invoke(prompt)

    unique_sources = []
    for doc in retrieved_docs:
        path = doc.metadata.get("source", "")
        file_name = os.path.basename(path)
        snippet = doc.page_content[:200]
        if file_name not in [src["file"] for src in unique_sources]:
            unique_sources.append({"file": file_name, "snippet": snippet})

    return {
        "answer": answer.strip(),
        "sources": unique_sources
    }

response = rag_with_sources("What is quantum decoherence?")
print("Answer:", response["answer"])
print("\nSources Used:")
for src in response["sources"]:
    print(f"- {src['file']}: {src['snippet']}...")


Answer: Quantum decoherence refers to the loss of quantum coherence in a system due to its interaction with the environment, resulting in a decrease in the ability to exhibit quantum behavior and an increase in classical-like behavior, leading to the apparent collapse of the wave function. (Source: [Document 3])

According to the concept of quantum mechanics, decoherence is a mechanism that explains how quantum systems interact with their environment, causing the loss of quantum coherence and leading to classical-like behavior. This process is different from actual wave-function collapse, as it does not generate wave-function collapse but rather provides a framework for its appearance (Source: [Document 1] and [Document 3])

In essence, decoherence is a way to understand why quantum systems begin to obey classical probability rules after interacting with their environment. It destroys interference effects and suppresses the quantum nature of the system, leading to a loss of quantum coh

Function enables an interactive chatbot using your RAG system.
It retrieves relevant context from your vector database, generates an answer using the LLM, and displays both the response and document sources.
It also keeps conversation history for more natural, ongoing dialogue.

In [29]:

chat_history = []

def ask_rag(question, k=4):
    retrieved_docs = db.similarity_search(question, k=k)

    context = ""
    for i, doc in enumerate(retrieved_docs, start=1):
        context += f"\n[Source {i}]\n{doc.page_content}\n"

    prompt = f"""
You are a RAG system trained on quantum computing and related data from Wikipedia.

Rules:
- Answer ONLY using the information provided in the context.
- Respond in clear, full sentences like ChatGPT.
- If the answer is not in the context, respond with: "I don't know."
- Do not output raw math formatting like |ψ⟩, ⊕, {{}}, or unnecessary LaTeX.
- Do not add personal introductions like "I am an AI."

Previous conversation:
{chat_history}

Context:
{context}

Question: {question}
Answer:
"""
    answer = llm.invoke(prompt).strip()

    chat_history.append({"user": question, "assistant": answer})

    sources = []
    for doc in retrieved_docs:
        file_name = os.path.basename(doc.metadata.get("source", ""))
        snippet = doc.page_content[:200].replace("\n", " ")
        if file_name not in [src["file"] for src in sources]:
            sources.append({"file": file_name, "snippet": snippet})

    print("\nAnswer:")
    print(answer)
    print("\nSources:")
    for i, src in enumerate(sources, start=1):
        print(f"{i}. {src['file']} → {src['snippet']}...")

while True:
    question = input("\nAsk a question (or type 'exit'): ")
    if question.lower() == "exit":
        break
    ask_rag(question)



Answer:
Quantum computing is a type of computation that uses quantum mechanical phenomena such as superposition and entanglement to perform calculations. Quantum computers have the potential to solve certain problems much faster than classical computers, but it's widely believed that they cannot solve all problems any faster. They can be viewed as sampling from quantum systems that evolve in ways that may be described as operating on an enormous number of possibilities simultaneously.

Sources:
1. Quantum_computing.txt → challenge by itself. Potential applications With focus on business management's point of view, the potential applications of quantum computing into four major categories are cybersecurity, data analyt...
2. Quantum_supremacy.txt → 'speed' of quantum computers, this research demonstrated their potential for 'stability' and 'reliability'. Furthermore, Google contributes to the open-source research ecosystem by providing software ...

Answer:
According to the provided so